In [ ]:
# Bloque 1: Preparación del entorno y del modelo (La IA se prepara para estudiar el dataset)

# 1. Importar las librerías necesarias
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# 2. Cargar el dataset 

df = pd.read_csv('adfa_ld_binary.csv')

print("Dimensiones del dataset:", df.shape)

# 3. Separar características (X) y la etiqueta (y) de forma automática
# X tomará TODAS las columnas MENOS la última
X = df.iloc[:, :-1]
# 'y' tomará ÚNICAMENTE la última columna
y = df.iloc[:, -1]

# Dividir en datos de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalar los datos (Crucial para Redes Neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Replicar la Arquitectura del Paper (Perceptrón Multicapa de 4 capas)
model = Sequential()
# Capa de entrada y primera capa oculta
model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
# Segunda capa oculta
model.add(Dense(32, activation='relu'))
# Capa de salida (sigmoid para clasificación binaria)
model.add(Dense(1, activation='sigmoid'))

# Compilar el modelo
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("¡Modelo MLP creado exitosamente y listo para entrenar!")

In [ ]:
# Bloque 2: Entrenamiento

print("Iniciando entrenamiento...")
history = model.fit(X_train_scaled, y_train, 
                    epochs=20,           # Le daremos 20 vueltas al dataset para que aprenda bien
                    batch_size=32,       # Procesará de 32 en 32 ejemplos
                    validation_split=0.2, # Usará un 20% de los datos para autoevaluarse mientras entrena
                    verbose=1)
print("\n¡Entrenamiento completado!")

In [ ]:
# Bloque 3: Evaluación del modelo

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Realizar predicciones con los datos de prueba (los que la IA no conoce)
y_pred_prob = model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype("int32")

# 2. Imprimir las métricas detalladas
print("### REPORTE DE CLASIFICACIÓN ###")
print(classification_report(y_test, y_pred))

# 3. Dibujar la Matriz de Confusión
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Ataque'], 
            yticklabels=['Normal', 'Ataque'])
plt.title('Matriz de Confusión del IDS')
plt.xlabel('Predicción de la IA')
plt.ylabel('Valor Real (Ground Truth)')
plt.show()

In [ ]:
# Bloque 4:LIME (Análisis de una instancia específica)
import lime
import lime.lime_tabular
import numpy as np

# 1. Configurar el explicador
nombres_caracteristicas = X.columns.tolist()
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=nombres_caracteristicas,
    class_names=['Normal', 'Ataque'],
    mode='classification'
)

# 2. Buscar un ataque real para analizar
indices_ataques = np.where(y_test == 1)[0]
idx_analizar = indices_ataques[0] 

# 3. EL ARREGLO: Función traductora para LIME
def predict_fn_lime(x):
    # Obtener la probabilidad de ataque de la IA
    p_ataque = model.predict(x)
    # Calcular la probabilidad normal (1 - prob_ataque)
    p_normal = 1 - p_ataque
    # Juntar ambas columnas para que LIME entienda que son dos clases
    return np.hstack((p_normal, p_ataque))

print(f"\nGenerando explicación LIME para el ataque en la fila {idx_analizar}...")

from IPython.display import HTML, display

# 4. Generar y mostrar la explicación 
exp = explainer.explain_instance(X_test_scaled[idx_analizar], predict_fn_lime, num_features=10)

# Extrae el HTML interno de LIME y se fuerza a mostrarse en VS Code
display(HTML(exp.as_html(show_table=True)))

In [ ]:
# Bloque 5: SHAP (Análisis Global) - VERSIÓN CAJA NEGRA 
import shap
import numpy as np

# 1. Crear una función predictora sencilla para SHAP
def f_predict_shap(x):
    # SHAP necesita que le devolvamos un arreglo plano de números
    return model.predict(x, verbose=0).flatten()

# 2. Resumir el "fondo" usando K-Means
# En lugar de usar datos crudos, creamos 25 "perfiles típicos" para que no tarde horas
background_summary = shap.kmeans(X_train_scaled, 25)

# 3. Usar KernelExplainer de SHAP, que es un método de caja negra
explainer_shap = shap.KernelExplainer(f_predict_shap, background_summary)

# 4. Calcular valores SHAP para 50 instancias de prueba
print("Calculando valores SHAP... ")
muestra_prueba = X_test_scaled[:50]
shap_values = explainer_shap.shap_values(muestra_prueba)

# 5. Generar el Gráfico de Resumen (Summary Plot)
print("\n¡Cálculo terminado! Generando gráfico...")
shap.summary_plot(shap_values, muestra_prueba, feature_names=X.columns.tolist())

In [ ]:
# Bloque 6: ANÁLISIS DE PERTURBACIÓN (Método Agresivo / Out-of-Distribution)
import numpy as np

cambios_etiqueta = 0
total = 50

print(f"🚨 INICIANDO ANÁLISIS MASIVO V3 (Perturbación Extrema - {total} instancias) 🚨\n")

for i in range(total):
    x_original = X_test_scaled[i].copy()
    
    # 1. Probabilidad original
    prob_original = model.predict(x_original.reshape(1, -1), verbose=0)[0][0]
    pred_original = int(prob_original > 0.5)
    
    # 2. Explicación LIME
    exp = explainer.explain_instance(x_original, predict_fn_lime, num_features=10)
    llave_lime = list(exp.local_exp.keys())[0]
    indices_importantes = [tupla[0] for tupla in exp.local_exp[llave_lime]]
    
    # 3. PERTURBACIÓN AGRESIVA (Rompiendo la señal)
    x_mod = x_original.copy()
    # Le asignamos un valor de -5 (5 desviaciones estándar lejos del promedio)
    # Esto simula una corrupción severa o la evasión total de esas 10 características
    x_mod[indices_importantes] = -5.0 
    
    # 4. Nueva probabilidad
    prob_nueva = model.predict(x_mod.reshape(1, -1), verbose=0)[0][0]
    pred_nueva = int(prob_nueva > 0.5)
    
    # 5. Debugging (Imprimimos solo los primeros 5 para ver el efecto)
    if i < 5:
        print(f"--- Instancia {i} ---")
        print(f"  Predicción: {pred_original} -> {pred_nueva}")
        print(f"  Confianza : {(prob_original*100):.2f}% -> {(prob_nueva*100):.2f}%")
        print(f"  Impacto en matriz (Suma del cambio): {np.sum(np.abs(x_original - x_mod)):.2f}\n")
    
    # 6. Registrar métricas
    if pred_original != pred_nueva:
        cambios_etiqueta += 1

print(f"📊 RESULTADOS GLOBALES V3 (Tasa de Evasión):")
print(f"Cambios de predicción logrados: {cambios_etiqueta}/{total}")
print(f"Porcentaje de éxito del ataque: {(cambios_etiqueta/total)*100:.2f}%")